In [1]:
# End-to-end topic + supervised pipeline with cross-checks

import os
import re
import json
import math
import time
import logging
import statistics
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Iterable, Optional, Any

import numpy as np
import pandas as pd

from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models.phrases import Phrases, Phraser
from gensim import corpora, models
from gensim.models.coherencemodel import CoherenceModel

from itertools import product

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix


try:
    import nltk
    from nltk import pos_tag, ne_chunk
    from nltk.corpus import wordnet
    from nltk.stem import WordNetLemmatizer
except ImportError:
    nltk = None
    pos_tag = None
    ne_chunk = None
    wordnet = None
    WordNetLemmatizer = None

#import matplotlib.pyplot as plt
#try:
#    import seaborn as sns
#except ImportError:  # seaborn is optional for the visuals section
#    sns = None


In [2]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('maxent_ne_chunker_tab')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sanie.s.rojas.lobo\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sanie.s.rojas.lobo\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     C:\Users\sanie.s.rojas.lobo\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!


True

In [3]:
# ----------------------- CONFIG -----------------------
CONFIG = {
    # Data
    "JSON_PATH": "1.1_wiki_results.json",
    "MANUAL_TAG_PATH": "manual_tag.csv",
    "MANUAL_TAG_SHEET": None,
    "MERGE_COMPANY_COLS": ["Company", "company", "Name", "name"],
    "MERGE_LABEL_COLS": ["Class", "class", "Label", "label", "Category", "category"],

    # Preprocessing controls
    "MIN_WORDS": 20,
    "ADDITIONAL_STOPWORDS": {
        "company", "companies", "business", "businesses", "enterprise", "enterprises", "firm", "group",
        "inc", "llc", "ltd", "corp", "sa", "ag", "srl", "nv", "se", "spa", "pte", "gmbh", "sasu",
        "bv", "ab", "oy", "kk", "plc", "co", "limited", "corporation"
    },
    "REMOVE_NUMERIC_TOKENS": True,
    "KEEP_ALPHABETIC_ONLY": True,
    "PHRASES_MIN_COUNT": 5,
    "PHRASES_THRESHOLD": 100.0,
    "ENABLE_POS_FILTER_STAGE": True,
    "POS_REMOVE_TAGS": ["PRP", "PRP$", "WP", "WP$", "WRB", "WDT", "DT", "PDT", "JJ", "JJR", "JJS"],
    "REMOVE_NAMED_ENTITIES": True,
    "NER_REMOVE_LABELS": ["PERSON", "ORG", "GPE", "LOC", "PRODUCT"],
    "ENABLE_LEMMATIZATION_STAGE": True,
    "LEMMATIZE_POS_OVERRIDES": {
        "J": "a",
        "V": "v",
        "N": "n",
        "R": "r"
    },


    # Dictionary filtering
    "DICT_NO_BELOW": 5,
    "DICT_NO_ABOVE": 0.95,
    "DICT_KEEP_N": 500000,

    # Logging controls
    "LOG_LEVEL": "INFO",
    "LOG_TIME_FORMAT": "%Y-%m-%d %H:%M:%S",
    "LOG_PRINT_TO_STDOUT": True,
    "LOG_INCLUDE_STAGE_STATS": True,
    "LOG_INCLUDE_MODEL_TIMINGS": True,

    # Metrics & outputs
    "COHERENCE_MEASURES": ["c_v", "u_mass", "c_npmi", "c_uci"],
    "TOP_WORDS_TOPN": 50,
    "FINAL_STAGE_NAME": "min_words",
    "OUTPUT_DIR": "artifacts",
    "SAVE_TOPICS_CSV": "company_topics_merged.csv",
    "SAVE_SUPERVISED_REPORT_CSV": "supervised_eval_report.csv",
    "SAVE_CLASS_TOPIC_SUMMARY_CSV_PREFIX": "class_topic_summary_",
    "STAGE_METRICS_FILENAME": "stage_metrics.csv",
    "TOPIC_COHERENCE_FILENAME": "topic_coherence_by_topic.csv",
    "TOP_WORDS_FILENAME": "top_words.csv",
    "RUN_LOG_FILENAME": "pipeline_report.txt",
    "TOP_WORDS_SUBDIR": "top_words",
    "VISUALS_SUBDIR": "visuals",

    # LDA params
    "LDA": {
        "num_topics": 25,
        "passes": 10,
        "iterations": 50,
        "random_state": 42,
        "chunksize": 500,
        "alpha": 0.85,
        "eta": 0.0001,
        "minimum_probability": 0.0
    },

    # HDP params
    "HDP_1": {
        "T": 25,
        "K": 3,
        "alpha": 0.85,
        "gamma": 1.0,
        "eta": 1,
        "scale": 1.0,
        "var_converge": 0.0001,
        "outputdir": None,
        "random_state": 42
    },
    "HDP_2": {
        "T": 100,
        "K": 3,
        "alpha": 0.85,
        "gamma": 1.0,
        "eta": 1,
        "scale": 1.0,
        "var_converge": 0.0001,
        "outputdir": None,
        "random_state": 42
    },

    # Supervised model evaluation
    "TEST_SIZE": 0.2,
    "RANDOM_STATE": 42,
    "SUPERVISED_SUBSETS": [
        {
            "name": "priority_industries",
            "labels": [
                "Industrial & Electronics",
                "Retail",
                "Aerospace & Defense",
                "Healthcare",
                "Financial Services"
            ],
            "min_docs": 20
        }
    ],


    # Topic model selection and tuning
    "BEST_MODEL_METRIC": "c_v",
    "RUN_TOPIC_TUNING": False,
    "LDA_TUNING_GRID": {
        "num_topics": [15, 25, 35],
        "passes": [5, 10],
        "alpha": ["symmetric", "auto"],
        "eta": ["auto", None]
    },
    "HDP_TUNING_GRID": {
        "T": [50, 100],
        "gamma": [0.5, 1.0]
    },

    # OpenAI integration (disabled by default)
    "ENABLE_OPENAI_CALLS": False,
    "OPENAI_MODEL": "gpt-4.1-mini",
    "OPENAI_TEMPERATURE": 0.2,
    "OPENAI_MAX_WORDS_PER_MODEL": 50,
    "INDUSTRY_LABELS": [
        "Advanced Electronics", "Aerospace and Defense", "Agriculture", "Automotive and Assembly",
        "Capital Projects and Infrastructure", "Chemicals", "Construction and Building Materials",
        "Consumer Packaged Goods", "Electric Power and Natural Gas", "Financial Services",
        "Healthcare Systems and Services", "High Tech", "Industrial and Electronics", "Media and Entertainment",
        "Metals and Mining", "Oil and Gas", "Pharmaceuticals and Medical Products", "Private Equity and Principal Investors",
        "Public and Social Sector", "Real Estate", "Retail", "Semiconductors", "Technology, Media and Telecommunications",
        "Travel, Logistics and Infrastructure", "Sustainability", "Professional Services"
    ],

    "OPENAI": {
        "API_KEY": None,
        "ORG_ID": None,
        "PROJECT": None
    },

    # Visualisation controls
    "ENABLE_VISUALS": False,
}


In [4]:
# --- Make output filenames unique with a timestamp ---
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_PREFIX = f"{RUN_ID}_"

output_dir = Path(CONFIG.get("OUTPUT_DIR", "artifacts"))
output_dir.mkdir(parents=True, exist_ok=True)

CONFIG["RUN_ID"] = RUN_ID
CONFIG["RUN_PREFIX"] = RUN_PREFIX
CONFIG["OUTPUT_DIR"] = output_dir

CONFIG["SAVE_TOPICS_CSV"] = output_dir / f"{RUN_PREFIX}company_topics_merged.csv"
CONFIG["SAVE_SUPERVISED_REPORT_CSV"] = output_dir / f"{RUN_PREFIX}supervised_eval_report.csv"
CONFIG["SAVE_CLASS_TOPIC_SUMMARY_CSV_PREFIX"] = f"{RUN_PREFIX}class_topic_summary_"

CONFIG["STAGE_METRICS_PATH"] = output_dir / f"{RUN_PREFIX}{CONFIG['STAGE_METRICS_FILENAME']}"
CONFIG["TOPIC_COHERENCE_PATH"] = output_dir / f"{RUN_PREFIX}{CONFIG['TOPIC_COHERENCE_FILENAME']}"
CONFIG["TOP_WORDS_PATH"] = output_dir / f"{RUN_PREFIX}{CONFIG['TOP_WORDS_FILENAME']}"
CONFIG["RUN_LOG_PATH"] = output_dir / f"{RUN_PREFIX}{CONFIG['RUN_LOG_FILENAME']}"

CONFIG["TOP_WORDS_DIR"] = output_dir / f"{RUN_PREFIX}{CONFIG['TOP_WORDS_SUBDIR']}"
CONFIG["TOP_WORDS_DIR"].mkdir(parents=True, exist_ok=True)

CONFIG["VISUALS_DIR"] = output_dir / f"{RUN_PREFIX}{CONFIG['VISUALS_SUBDIR']}"
CONFIG["VISUALS_DIR"].mkdir(parents=True, exist_ok=True)

print("Using output paths:")
print(" - Topics CSV:", CONFIG["SAVE_TOPICS_CSV"])
print(" - Supervised report CSV:", CONFIG["SAVE_SUPERVISED_REPORT_CSV"])
print(" - Stage metrics CSV:", CONFIG["STAGE_METRICS_PATH"])
print(" - Topic coherence CSV:", CONFIG["TOPIC_COHERENCE_PATH"])
print(" - Top words CSV:", CONFIG["TOP_WORDS_PATH"])
print(" - Log file:", CONFIG["RUN_LOG_PATH"])
print(" - Top words dir:", CONFIG["TOP_WORDS_DIR"])
print(" - Visuals dir:", CONFIG["VISUALS_DIR"])


Using output paths:
 - Topics CSV: artifacts\20250921-205356_company_topics_merged.csv
 - Supervised report CSV: artifacts\20250921-205356_supervised_eval_report.csv
 - Stage metrics CSV: artifacts\20250921-205356_stage_metrics.csv
 - Topic coherence CSV: artifacts\20250921-205356_topic_coherence_by_topic.csv
 - Top words CSV: artifacts\20250921-205356_top_words.csv
 - Log file: artifacts\20250921-205356_pipeline_report.txt
 - Top words dir: artifacts\20250921-205356_top_words
 - Visuals dir: artifacts\20250921-205356_visuals


In [5]:
# ---------------------- HELPERS -----------------------

@dataclass
class PreprocessStageResult:
    name: str
    description: str
    tokens: List[List[str]]
    df: pd.DataFrame
    mask: Optional[np.ndarray] = None
    dictionary: Optional[corpora.Dictionary] = None
    corpus: Optional[List] = None
    token_stats: Dict[str, Any] = field(default_factory=dict)
    meta: Dict[str, Any] = field(default_factory=dict)


@dataclass
class TopicModelRun:
    stage: str
    model_key: str
    label: str
    model: Optional[object]
    lda_like: Optional[object]
    train_seconds: float
    metrics: Dict[str, float] = field(default_factory=dict)
    per_topic_coherence: Dict[str, List[float]] = field(default_factory=dict)
    perplexity: float = float("nan")
    top_words: List[List[str]] = field(default_factory=list)
    dictionary_size: int = 0
    doc_count: int = 0
    notes: Optional[str] = None
    stage_stats: Dict[str, Any] = field(default_factory=dict)



class PipelineLogger:
    LEVELS = {
        "NOTSET": logging.NOTSET,
        "DEBUG": logging.DEBUG,
        "INFO": logging.INFO,
        "WARNING": logging.WARNING,
        "WARN": logging.WARNING,
        "ERROR": logging.ERROR,
        "CRITICAL": logging.CRITICAL,
    }

    def __init__(self, cfg: Dict[str, Any], sink: Optional[List[str]] = None):
        self.cfg = cfg or {}
        self.lines = sink if sink is not None else []
        self.time_fmt = self.cfg.get("LOG_TIME_FORMAT", "%Y-%m-%d %H:%M:%S")
        level_name = str(self.cfg.get("LOG_LEVEL", "INFO")).upper()
        self.level = self.LEVELS.get(level_name, logging.INFO)
        self.echo = bool(self.cfg.get("LOG_PRINT_TO_STDOUT", True))

    def log(self, message: str, level: str = "INFO", extra: Optional[Dict[str, Any]] = None):
        level_name = level.upper()
        level_value = self.LEVELS.get(level_name, logging.INFO)
        if level_value < self.level:
            return
        timestamp = time.strftime(self.time_fmt)
        text = f"[{timestamp}] [{level_name}] {message}"
        if extra:
            extra_text = format_log_extra(extra)
            if extra_text:
                text += f" | {extra_text}"
        if not text.endswith(""):
            text += ""
        self.lines.append(text)
        if self.echo:
            print(text.rstrip())


def format_log_extra(extra: Dict[str, Any]) -> str:
    parts = []
    for key, value in extra.items():
        if value is None:
            continue
        if isinstance(value, (float, np.floating)):
            parts.append(f"{key}={float(value):.4f}")
        elif isinstance(value, (int, np.integer)):
            parts.append(f"{key}={int(value)}")
        else:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


def ensure_nltk_resource(resource: str, download_name: Optional[str] = None, logger: Optional[PipelineLogger] = None) -> bool:
    if nltk is None:
        return False
    try:
        nltk.data.find(resource)
        return True
    except LookupError:
        name = download_name or resource.split("/")[-1]
        try:
            nltk.download(name, quiet=True)
            nltk.data.find(resource)
            if logger:
                logger.log(f"Downloaded NLTK resource '{name}'", level="DEBUG")
            return True
        except Exception as exc:
            if logger:
                logger.log(f"Could not access NLTK resource '{resource}': {exc}", level="WARNING")
            return False


def ensure_pos_tagger(logger: Optional[PipelineLogger] = None) -> bool:
    if nltk is None:
        return False
    if ensure_nltk_resource("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng", logger):
        return True
    return ensure_nltk_resource("taggers/averaged_perceptron_tagger", "averaged_perceptron_tagger", logger)


def ensure_ner_resources(logger: Optional[PipelineLogger] = None) -> bool:
    if nltk is None:
        return False
    resources = [
        ("chunkers/maxent_ne_chunker", "maxent_ne_chunker"),
        ("corpora/words", "words"),
    ]
    ok = True
    for res, name in resources:
        ok = ensure_nltk_resource(res, name, logger) and ok
    return ok


def ensure_wordnet(logger: Optional[PipelineLogger] = None) -> bool:
    if nltk is None:
        return False
    resources = [
        ("corpora/wordnet", "wordnet"),
        ("corpora/omw-1.4", "omw-1.4"),
    ]
    ok = True
    for res, name in resources:
        ok = ensure_nltk_resource(res, name, logger) and ok
    return ok


def compute_token_statistics(tokens_list: List[List[str]]) -> Dict[str, Any]:
    lengths = [len(toks) for toks in tokens_list]
    doc_count = len(lengths)
    non_empty = sum(1 for length in lengths if length > 0)
    total_tokens = int(sum(lengths))
    avg_tokens = float(total_tokens / non_empty) if non_empty else 0.0
    median_tokens = float(statistics.median(lengths)) if lengths else 0.0
    max_tokens = max(lengths, default=0)
    min_tokens = min(lengths, default=0)
    std_tokens = float(np.std(lengths)) if lengths else 0.0
    unique_tokens = len({token for doc in tokens_list for token in doc})
    return {
        "documents": doc_count,
        "non_empty_docs": non_empty,
        "total_tokens": total_tokens,
        "avg_tokens": avg_tokens,
        "median_tokens": median_tokens,
        "min_tokens": min_tokens,
        "max_tokens": max_tokens,
        "std_tokens": std_tokens,
        "unique_tokens": unique_tokens,
    }


def get_wordnet_pos(tag: str, overrides: Optional[Dict[str, str]] = None) -> Optional[str]:
    overrides = overrides or {}
    for prefix, wn_tag in overrides.items():
        if tag.startswith(prefix):
            return wn_tag
    if tag.startswith("J"):
        return wordnet.ADJ if wordnet else "a"
    if tag.startswith("V"):
        return wordnet.VERB if wordnet else "v"
    if tag.startswith("N"):
        return wordnet.NOUN if wordnet else "n"
    if tag.startswith("R"):
        return wordnet.ADV if wordnet else "r"
    return None


def filter_tokens_with_pos_ner(tokens_list: List[List[str]], cfg: Dict, logger: Optional[PipelineLogger] = None):
    info = {
        "removed_pos_tokens": 0,
        "removed_entity_tokens": 0,
        "documents_modified": 0,
        "skipped": False,
    }
    if not cfg.get("ENABLE_POS_FILTER_STAGE", False):
        info["skipped"] = "disabled"
        info["summary"] = "POS/NER filtering disabled via config"
        return tokens_list, info
    if nltk is None or pos_tag is None:
        info["skipped"] = "nltk_missing"
        info["summary"] = "NLTK not available; POS/NER filtering skipped"
        if logger:
            logger.log(info["summary"], level="WARNING")
        return tokens_list, info
    if not ensure_pos_tagger(logger):
        info["skipped"] = "tagger_missing"
        info["summary"] = "NLTK POS tagger unavailable; stage skipped"
        return tokens_list, info

    remove_entities = bool(cfg.get("REMOVE_NAMED_ENTITIES", False))
    ner_ready = True
    if remove_entities:
        ner_ready = ensure_ner_resources(logger)
        if not ner_ready and logger:
            logger.log("Named-entity resources unavailable; continuing without NER filtering", level="WARNING")

    remove_tags = set(cfg.get("POS_REMOVE_TAGS", []))
    filtered_tokens = []
    for toks in tokens_list:
        if not toks:
            filtered_tokens.append([])
            continue
        try:
            tagged = pos_tag(toks)
        except LookupError as exc:
            info["skipped"] = f"tagger_error: {exc}"
            info["summary"] = "POS tagging failed; returning original tokens"
            if logger:
                logger.log(info["summary"], level="ERROR")
            return tokens_list, info

        entity_words = set()
        if remove_entities and ner_ready:
            try:
                tree = ne_chunk(tagged, binary=False)
                wanted = set(cfg.get("NER_REMOVE_LABELS", []))
                for chunk in tree:
                    if hasattr(chunk, "label") and chunk.label() in wanted:
                        entity_words.update(word for word, _ in chunk.leaves())
            except LookupError as exc:
                if logger:
                    logger.log(f"NER chunker missing ({exc}); skipping NER filtering", level="WARNING")
                entity_words = set()

        doc_filtered = []
        removed_pos = 0
        removed_ner = 0
        for word, tag in tagged:
            base_tag = tag.split("-")[0]
            if tag in remove_tags or base_tag in remove_tags:
                removed_pos += 1
                continue
            if entity_words and word in entity_words:
                removed_ner += 1
                continue
            doc_filtered.append(word)
        if removed_pos or removed_ner:
            info["documents_modified"] += 1
        info["removed_pos_tokens"] += removed_pos
        info["removed_entity_tokens"] += removed_ner
        filtered_tokens.append(doc_filtered)

    removed_total = info["removed_pos_tokens"] + info["removed_entity_tokens"]
    info["summary"] = (
        f"Removed {removed_total} tokens via POS/NER filtering "
        f"(POS={info['removed_pos_tokens']}, NER={info['removed_entity_tokens']}), "
        f"docs_modified={info['documents_modified']}"
    )
    info["stats"] = {
        "removed_pos_tokens": info["removed_pos_tokens"],
        "removed_entity_tokens": info["removed_entity_tokens"],
        "documents_modified": info["documents_modified"],
    }
    return filtered_tokens, info


def lemmatize_token_sequences(tokens_list: List[List[str]], cfg: Dict, logger: Optional[PipelineLogger] = None):
    info = {
        "changed_tokens": 0,
        "documents_modified": 0,
        "skipped": False,
    }
    if not cfg.get("ENABLE_LEMMATIZATION_STAGE", False):
        info["skipped"] = "disabled"
        info["summary"] = "Lemmatization disabled via config"
        return tokens_list, info
    if WordNetLemmatizer is None:
        info["skipped"] = "wordnet_lemmatizer_missing"
        info["summary"] = "WordNet lemmatizer unavailable; skipping stage"
        if logger:
            logger.log(info["summary"], level="WARNING")
        return tokens_list, info
    if not ensure_pos_tagger(logger):
        info["skipped"] = "tagger_missing"
        info["summary"] = "NLTK POS tagger unavailable; cannot lemmatize"
        return tokens_list, info
    if not ensure_wordnet(logger):
        info["skipped"] = "wordnet_missing"
        info["summary"] = "WordNet corpora unavailable; skipping stage"
        return tokens_list, info

    lemmatizer = WordNetLemmatizer()
    overrides = cfg.get("LEMMATIZE_POS_OVERRIDES", {})
    outputs = []
    for toks in tokens_list:
        if not toks:
            outputs.append([])
            continue
        try:
            tagged = pos_tag(toks)
        except LookupError as exc:
            info["skipped"] = f"tagger_error: {exc}"
            info["summary"] = "POS tagging failed during lemmatization; returning original tokens"
            if logger:
                logger.log(info["summary"], level="ERROR")
            return tokens_list, info
        doc_changed = False
        doc_tokens = []
        for word, tag in tagged:
            wn_tag = get_wordnet_pos(tag, overrides)
            lemma = lemmatizer.lemmatize(word, wn_tag) if wn_tag else lemmatizer.lemmatize(word)
            if lemma != word:
                info["changed_tokens"] += 1
                doc_changed = True
            doc_tokens.append(lemma)
        if doc_changed:
            info["documents_modified"] += 1
        outputs.append(doc_tokens)

    info["summary"] = (
        f"Lemmatized tokens with {info['changed_tokens']} replacements "
        f"across {info['documents_modified']} documents"
    )
    info["stats"] = {
        "lemmatized_tokens": info["changed_tokens"],
        "documents_modified": info["documents_modified"],
    }
    return outputs, info

def normalize_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def load_json_dataset(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        items = list(data.items())
        df = pd.DataFrame(items, columns=["Company", "Description"])
    elif isinstance(data, list):
        df = pd.DataFrame(data)
        if "Description" not in df.columns:
            text_cols = [c for c in df.columns if "desc" in c.lower()]
            if text_cols:
                df = df.rename(columns={text_cols[0]: "Description"})
        if "Company" not in df.columns:
            name_cols = [c for c in df.columns if "name" in c.lower()]
            if name_cols:
                df = df.rename(columns={name_cols[0]: "Company"})
        if "Company" not in df.columns or "Description" not in df.columns:
            raise ValueError("Could not infer 'Company' and 'Description' columns from JSON file.")
    else:
        raise ValueError("Unsupported JSON structure. Expect dict{name: description} or list of records.")
    return df


def clean_and_tokenize(doc: str, keep_alpha_only: bool = True) -> List[str]:
    tokens = simple_preprocess(doc or "", deacc=True, min_len=2, max_len=80)
    if keep_alpha_only:
        tokens = [t for t in tokens if t.isalpha()]
    return tokens


def build_ngrams(tokens_list: List[List[str]], min_count: int = 5, threshold: float = 100.0) -> List[List[str]]:
    if not tokens_list:
        return tokens_list
    bigram = Phrases(tokens_list, min_count=min_count, threshold=threshold)
    trigram = Phrases(bigram[tokens_list], threshold=threshold)
    bigram_mod = Phraser(bigram)
    trigram_mod = Phraser(trigram)
    out = []
    for toks in tokens_list:
        bi = bigram_mod[toks]
        tri = trigram_mod[bi]
        out.append(tri)
    return out



def build_preprocessing_stages(df: pd.DataFrame, cfg: Dict, logger: Optional[PipelineLogger] = None) -> List[PreprocessStageResult]:
    df_stage = df.reset_index(drop=True).copy()
    descriptions = df_stage["Description"].astype(str).tolist()
    tokens_tokenized = [
        clean_and_tokenize(text, keep_alpha_only=cfg.get("KEEP_ALPHABETIC_ONLY", True))
        for text in descriptions
    ]
    stages: List[PreprocessStageResult] = [
        PreprocessStageResult(
            name="tokenized",
            description="Tokenized (lowercased, accent stripped)",
            tokens=tokens_tokenized,
            df=df_stage,
            token_stats=compute_token_statistics(tokens_tokenized),
        )
    ]

    if cfg.get("REMOVE_NUMERIC_TOKENS", False):
        tokens_no_num = [
            [t for t in toks if not any(ch.isdigit() for ch in t)]
            for toks in tokens_tokenized
        ]
        desc_numeric = "Removed tokens that contain digits"
    else:
        tokens_no_num = tokens_tokenized
        desc_numeric = "Numeric token removal disabled; tokens unchanged"
    stages.append(
        PreprocessStageResult(
            name="no_numeric",
            description=desc_numeric,
            tokens=tokens_no_num,
            df=df_stage,
            token_stats=compute_token_statistics(tokens_no_num),
        )
    )

    stopwords = set(STOPWORDS).union(cfg.get("ADDITIONAL_STOPWORDS", set()))
    tokens_no_stop = [[t for t in toks if t not in stopwords] for toks in tokens_no_num]
    stages.append(
        PreprocessStageResult(
            name="no_stopwords",
            description="Removed default and custom stop words",
            tokens=tokens_no_stop,
            df=df_stage,
            token_stats=compute_token_statistics(tokens_no_stop),
        )
    )

    pos_tokens, pos_info = filter_tokens_with_pos_ner(tokens_no_stop, cfg, logger)
    pos_desc = pos_info.get("summary", "POS/NER filtering applied")
    stages.append(
        PreprocessStageResult(
            name="pos_filtered",
            description=pos_desc,
            tokens=pos_tokens,
            df=df_stage,
            token_stats=compute_token_statistics(pos_tokens),
            meta={"pos_info": pos_info},
        )
    )

    lemma_tokens, lemma_info = lemmatize_token_sequences(pos_tokens, cfg, logger)
    lemma_desc = lemma_info.get("summary", "Lemmatization applied")
    stages.append(
        PreprocessStageResult(
            name="lemmatized",
            description=lemma_desc,
            tokens=lemma_tokens,
            df=df_stage,
            token_stats=compute_token_statistics(lemma_tokens),
            meta={"lemmatize_info": lemma_info},
        )
    )

    tokens_ngrams = build_ngrams(
        lemma_tokens,
        min_count=cfg.get("PHRASES_MIN_COUNT", 5),
        threshold=cfg.get("PHRASES_THRESHOLD", 100.0),
    )
    stages.append(
        PreprocessStageResult(
            name="with_ngrams",
            description="Added bigram and trigram phrases",
            tokens=tokens_ngrams,
            df=df_stage,
            token_stats=compute_token_statistics(tokens_ngrams),
        )
    )

    min_words = cfg.get("MIN_WORDS", 0)
    if min_words and min_words > 0:
        mask = np.array([len(toks) >= min_words for toks in tokens_ngrams], dtype=bool)
        filtered_df = df_stage.loc[mask].reset_index(drop=True)
        filtered_tokens = [t for t, keep in zip(tokens_ngrams, mask) if keep]
        desc_min_words = f"Kept documents with >= {min_words} tokens ({mask.sum()} of {len(mask)})"
    else:
        mask = None
        filtered_df = df_stage
        filtered_tokens = tokens_ngrams
        desc_min_words = "MIN_WORDS disabled; using n-gram tokens"
    stages.append(
        PreprocessStageResult(
            name="min_words",
            description=desc_min_words,
            tokens=filtered_tokens,
            df=filtered_df,
            mask=mask,
            token_stats=compute_token_statistics(filtered_tokens),
        )
    )

    return stages


def make_dictionary_corpus(texts: List[List[str]], cfg: Dict):
    if not texts:
        return None, []
    dictionary = corpora.Dictionary(texts)
    if len(dictionary) > 0:
        dictionary.filter_extremes(
            no_below=cfg.get("DICT_NO_BELOW", 5),
            no_above=cfg.get("DICT_NO_ABOVE", 0.95),
            keep_n=cfg.get("DICT_KEEP_N", 200000),
        )
    corpus = [dictionary.doc2bow(doc) for doc in texts]
    return dictionary, corpus


def attach_dictionary_and_corpus(stage: PreprocessStageResult, cfg: Dict) -> PreprocessStageResult:
    dictionary, corpus = make_dictionary_corpus(stage.tokens, cfg)
    stage.dictionary = dictionary
    stage.corpus = corpus
    return stage


def train_lda(corpus, dictionary, cfg):
    if dictionary is None or not corpus:
        raise ValueError("Empty corpus passed to LDA")
    lda = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=cfg["num_topics"],
        passes=cfg["passes"],
        iterations=cfg["iterations"],
        random_state=cfg["random_state"],
        chunksize=cfg["chunksize"],
        alpha=cfg["alpha"],
        eta=cfg["eta"],
        minimum_probability=cfg.get("minimum_probability", 0.0),
    )
    return lda


def train_hdp(corpus, dictionary, cfg):
    if dictionary is None or not corpus:
        raise ValueError("Empty corpus passed to HDP")
    hdp = models.HdpModel(
        corpus=corpus,
        id2word=dictionary,
        T=cfg["T"],
        K=cfg["K"],
        alpha=cfg["alpha"],
        gamma=cfg["gamma"],
        eta=cfg["eta"],
        scale=cfg["scale"],
        var_converge=cfg["var_converge"],
        outputdir=cfg["outputdir"],
        random_state=cfg.get("random_state"),
    )
    return hdp


def as_lda(model):
    return model.suggested_lda_model() if hasattr(model, "suggested_lda_model") else model


def topic_word_lists(model_like, topn: int = 50) -> List[List[str]]:
    if model_like is None:
        return []
    topics = model_like.show_topics(num_topics=-1, num_words=topn, formatted=False)
    topics = sorted(topics, key=lambda x: x[0])
    return [[w for (w, _w) in words] for (_tid, words) in topics]


def compute_coherence_scores(model_like, texts, dictionary, corpus, measures) -> Tuple[Dict[str, float], Dict[str, List[float]]]:
    metrics = {}
    per_topic = {}
    for measure in measures:
        try:
            kwargs = {"dictionary": dictionary, "coherence": measure}
            if measure == "u_mass":
                kwargs["corpus"] = corpus
            else:
                kwargs["texts"] = texts
            if isinstance(model_like, list):
                kwargs["topics"] = model_like
                cm = CoherenceModel(**kwargs)
            else:
                kwargs["model"] = model_like
                cm = CoherenceModel(**kwargs)
            metrics[measure] = float(cm.get_coherence())
            per_topic[measure] = [float(x) for x in cm.get_coherence_per_topic()]
        except Exception:
            metrics[measure] = float("nan")
            per_topic[measure] = []
    return metrics, per_topic


def lda_perplexity(lda_like, corpus):
    if lda_like is None or not corpus:
        return float("nan")
    try:
        return float(2 ** (-lda_like.log_perplexity(corpus)))
    except Exception:
        return float("nan")



def _empty_run(stage: PreprocessStageResult, model_key: str, label: str, message: str) -> TopicModelRun:
    return TopicModelRun(
        stage=stage.name,
        model_key=model_key,
        label=label,
        model=None,
        lda_like=None,
        train_seconds=float("nan"),
        metrics={"error": message},
        per_topic_coherence={},
        perplexity=float("nan"),
        top_words=[],
        dictionary_size=len(stage.dictionary) if stage.dictionary else 0,
        doc_count=len(stage.tokens),
        notes=message,
        stage_stats=stage.token_stats,
    )







def train_topic_models_for_stage(stage: PreprocessStageResult, cfg: Dict) -> List[TopicModelRun]:
    measures = cfg.get("COHERENCE_MEASURES", ["c_v"])
    topn = cfg.get("TOP_WORDS_TOPN", 50)
    if stage.dictionary is None or len(stage.dictionary) == 0 or not stage.corpus:
        message = "Empty corpus/dictionary after preprocessing"
        return [
            _empty_run(stage, model_key, label, message)
            for model_key, label in [("LDA", "LDA"), ("HDP_1", "HDP1"), ("HDP_2", "HDP2")]
        ]

    specs = [
        ("LDA", "LDA", cfg["LDA"], train_lda),
        ("HDP_1", "HDP1", cfg["HDP_1"], train_hdp),
        ("HDP_2", "HDP2", cfg["HDP_2"], train_hdp),
    ]
    runs: List[TopicModelRun] = []
    for model_key, label, model_cfg, trainer in specs:
        try:
            t0 = time.time()
            model = trainer(stage.corpus, stage.dictionary, model_cfg)
            train_seconds = time.time() - t0
            lda_like = as_lda(model)
            top_words = topic_word_lists(lda_like, topn=topn)
            metrics, per_topic = compute_coherence_scores(
                lda_like,
                stage.tokens,
                stage.dictionary,
                stage.corpus,
                measures,
            )
            perplexity = lda_perplexity(lda_like, stage.corpus)
            notes = None
        except Exception as exc:
            runs.append(_empty_run(stage, model_key, label, str(exc)))
            continue

        runs.append(
            TopicModelRun(
                stage=stage.name,
                model_key=model_key,
                label=label,
                model=model,
                lda_like=lda_like,
                train_seconds=train_seconds,
                metrics=metrics,
                per_topic_coherence=per_topic,
                perplexity=perplexity,
                top_words=top_words,
                dictionary_size=len(stage.dictionary),
                doc_count=len(stage.tokens),
                notes=notes,
                stage_stats=stage.token_stats,
            )
        )
    return runs






def format_metric(value) -> str:
    try:
        if value is None:
            return "nan"
        if isinstance(value, (float, np.floating)) and (np.isnan(value) or np.isinf(value)):
            return "nan"
        return f"{float(value):.4f}"
    except Exception:
        return "nan"



def run_topic_models_across_stages(stages: List[PreprocessStageResult], cfg: Dict, logger: Optional[PipelineLogger] = None):
    logger = logger or PipelineLogger(cfg)
    stage_runs = {}
    metrics_rows = []
    per_topic_rows = []
    top_words_rows = []

    for stage in stages:
        attach_dictionary_and_corpus(stage, cfg)
        vocab_size = len(stage.dictionary) if stage.dictionary is not None else 0
        stage_stats = stage.token_stats or compute_token_statistics(stage.tokens)
        stage_stats = {**stage_stats, "vocab_size": vocab_size}

        logger.log(
            f"[Stage {stage.name}] {stage.description}",
            extra=stage_stats if cfg.get("LOG_INCLUDE_STAGE_STATS", True) else None,
        )

        stage_t0 = time.time()
        runs = train_topic_models_for_stage(stage, cfg)
        stage_duration = time.time() - stage_t0
        if cfg.get("LOG_INCLUDE_MODEL_TIMINGS", True):
            logger.log(
                f"[Stage {stage.name}] topic modelling finished in {stage_duration:.2f}s",
                level="DEBUG",
                extra={"stage_train_seconds": stage_duration},
            )

        for run in runs:
            stage_runs[(stage.name, run.model_key)] = run
            if run.model is None:
                logger.log(f"[Stage {stage.name}::{run.label}] {run.notes}", level="WARNING")
                continue

            metric_parts = {f"coherence_{m}": run.metrics.get(m) for m in cfg.get("COHERENCE_MEASURES", [])}
            extra = {
                "train_seconds": run.train_seconds,
                "perplexity": run.perplexity,
                "doc_count": run.doc_count,
                "dictionary_size": run.dictionary_size,
                **metric_parts,
            }
            metric_text = " ".join(f"{k}={format_metric(v)}" for k, v in metric_parts.items())
            logger.log(
                f"[Stage {stage.name}::{run.label}] train_s={run.train_seconds:.2f}s perplexity={format_metric(run.perplexity)} {metric_text}",
                extra=extra,
            )

            metrics_row = {
                "run_id": cfg["RUN_ID"],
                "stage": stage.name,
                "stage_description": stage.description,
                "model": run.label,
                "train_seconds": run.train_seconds,
                "doc_count": run.doc_count,
                "dictionary_size": run.dictionary_size,
                "perplexity": run.perplexity,
                "stage_train_seconds": stage_duration,
            }
            metrics_row.update(stage_stats)
            for m in cfg.get("COHERENCE_MEASURES", []):
                metrics_row[f"coherence_{m}"] = run.metrics.get(m, float("nan"))
            metrics_rows.append(metrics_row)

            for metric_name, scores in run.per_topic_coherence.items():
                for idx, score in enumerate(scores):
                    per_topic_rows.append(
                        {
                            "run_id": cfg["RUN_ID"],
                            "stage": stage.name,
                            "model": run.label,
                            "topic_index": idx,
                            "metric": metric_name,
                            "score": score,
                        }
                    )

            for topic_idx, words in enumerate(run.top_words):
                top_limit = cfg.get("TOP_WORDS_TOPN", 50)
                for rank, word in enumerate(words[: top_limit], start=1):
                    top_words_rows.append(
                        {
                            "run_id": cfg["RUN_ID"],
                            "stage": stage.name,
                            "model": run.label,
                            "topic_index": topic_idx,
                            "word_rank": rank,
                            "word": word,
                        }
                    )

    metrics_df = pd.DataFrame(metrics_rows)
    per_topic_df = pd.DataFrame(per_topic_rows)
    top_words_df = pd.DataFrame(top_words_rows)
    return stage_runs, metrics_df, per_topic_df, top_words_df, logger.lines






def select_best_topic_model(metrics_df: pd.DataFrame, metric: str = "c_v"):
    if metrics_df.empty:
        return None
    metric_col = f"coherence_{metric}"
    if metric_col not in metrics_df.columns:
        return None
    df_numeric = metrics_df.dropna(subset=[metric_col])
    if df_numeric.empty:
        return None
    best_idx = df_numeric[metric_col].astype(float).idxmax()
    return df_numeric.loc[best_idx].to_dict()


def save_stage_outputs(metrics_df, per_topic_df, top_words_df, cfg: Dict):
    if not metrics_df.empty:
        metrics_df.to_csv(CONFIG["STAGE_METRICS_PATH"], index=False)
    if not per_topic_df.empty:
        per_topic_df.to_csv(CONFIG["TOPIC_COHERENCE_PATH"], index=False)
    if not top_words_df.empty:
        top_words_df.to_csv(CONFIG["TOP_WORDS_PATH"], index=False)
        top_words_dir = Path(CONFIG["TOP_WORDS_DIR"])
        for (stage, model), group in top_words_df.groupby(["stage", "model"]):
            out_path = top_words_dir / f"{CONFIG['RUN_PREFIX']}{stage}_{model}_top_words.csv"
            group.sort_values(["topic_index", "word_rank"]).to_csv(out_path, index=False)



def write_log(lines: List[str], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    content = "".join(line if line.endswith("") else f"{line}" for line in lines)
    with path.open("w", encoding="utf-8") as f:
        f.write(content)





def dense_doc_topics(model, corpus, cap_topics=None):
    use_model = as_lda(model)
    k = None
    try:
        k = use_model.get_topics().shape[0]
    except Exception:
        pass
    if k is None or k == 0:
        try:
            dist0 = use_model.get_document_topics(corpus[0], minimum_probability=0)
        except (AttributeError, IndexError):
            dist0 = []
        k = max((tid for tid, _ in dist0), default=-1) + 1
    if cap_topics is not None:
        k = cap_topics
    M = np.zeros((len(corpus), k), dtype=float)
    for i, bow in enumerate(corpus):
        try:
            dist = use_model.get_document_topics(bow, minimum_probability=0)
        except AttributeError:
            dist = use_model[bow]
        for tid, prob in dist:
            if 0 <= tid < k:
                M[i, tid] = float(prob)
    return M, k


def autodetect_column(df: pd.DataFrame, candidates: List[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    raise ValueError(f"None of {candidates} found in columns: {df.columns.tolist()}")


def load_manual_tags(path, sheet=None, company_cands=None, label_cands=None):
    if path.lower().endswith(".xlsx"):
        tags = pd.read_excel(path, sheet_name=sheet)
    elif path.lower().endswith(".csv"):
        tags = pd.read_csv(path)
    else:
        raise ValueError("Unsupported manual tag file format.")
    company_col = autodetect_column(tags, company_cands or CONFIG["MERGE_COMPANY_COLS"])
    label_col = autodetect_column(tags, label_cands or CONFIG["MERGE_LABEL_COLS"])
    tags = tags.rename(columns={company_col: "Company_Tag", label_col: "Manual_Tag"})
    return tags[["Company_Tag", "Manual_Tag"]]


def filter_labels_min_count(X, y, min_count=2, merge_to_other=False, other_label="Other"):
    vc = pd.Series(y).value_counts()
    rare = vc[vc < min_count].index.tolist()
    if not rare:
        return X, y, None, np.ones(len(y), dtype=bool)
    if merge_to_other:
        y2 = np.array([other_label if t in rare else t for t in y], dtype=object)
        mask = np.ones(len(y), dtype=bool)
        return X, y2, {"merged": rare, "min_count": min_count}, mask
    mask = ~np.isin(y, rare)
    X2, y2 = X[mask], y[mask]
    return X2, y2, {"dropped": rare, "min_count": min_count}, mask


def pick_best_supervised(X, y, random_state=42, test_size=None):
    X2, y2, info, mask = filter_labels_min_count(
        X, y, min_count=2, merge_to_other=True, other_label="Other"
    )
    if info:
        print("Supervised label adjustment:", info)
    uniq = np.unique(y2)
    if len(uniq) < 2 or len(y2) < 10:
        raise ValueError("Not enough labeled data after filtering to run supervised evaluation.")
    if test_size is None:
        test_size = CONFIG["TEST_SIZE"]

    models = {
        "LogReg_balanced": make_pipeline(
            StandardScaler(with_mean=False),
            LogisticRegression(max_iter=2000, class_weight="balanced"),
        ),
        "LinearSVC_balanced": make_pipeline(
            StandardScaler(with_mean=False),
            LinearSVC(class_weight="balanced"),
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            n_jobs=-1,
            random_state=random_state,
            class_weight="balanced_subsample",
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=random_state),
    }

    X_train, X_test, y_train, y_test = train_test_split(
        X2, y2, test_size=test_size, random_state=random_state, stratify=y2
    )

    results = []
    best_name, best_model, best_score = None, None, -1.0
    for name, clf in models.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        f1 = f1_score(y_test, y_pred, average="macro")
        acc = accuracy_score(y_test, y_pred)
        results.append((name, f1, acc, clf, y_test, y_pred))
        if f1 > best_score:
            best_score, best_name, best_model = f1, name, clf

    report_rows = [
        {"Model": n, "MacroF1": f1, "Accuracy": acc}
        for n, f1, acc, *_ in results
    ]
    report_df = pd.DataFrame(report_rows).sort_values("MacroF1", ascending=False).reset_index(drop=True)

    y_test_best = [r[4] for r in results if r[0] == best_name][0]
    y_pred_best = [r[5] for r in results if r[0] == best_name][0]

    report_txt = classification_report(y_test_best, y_pred_best, digits=4)
    cm = confusion_matrix(y_test_best, y_pred_best)

    return best_name, best_model, report_df, report_txt, cm, (X_train, X_test, y_train, y_test, y_pred_best), mask


def class_topic_summary(df, topic_cols, label_col="Manual_Tag", topn=5):
    means = df.groupby(label_col)[topic_cols].mean()
    summary = []
    for cls in means.index:
        row = means.loc[cls]
        top_topics = row.sort_values(ascending=False).head(topn)
        for t, v in top_topics.items():
            summary.append({"Class": cls, "Topic": t, "AvgWeight": v})
    return pd.DataFrame(summary)


def expand_param_grid(grid: Dict[str, Iterable]):
    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    for combo in product(*values):
        yield dict(zip(keys, combo))


def grid_search_lda(stage: PreprocessStageResult, cfg: Dict, grid: Dict[str, Iterable]):
    if stage.dictionary is None or len(stage.dictionary) == 0 or not stage.corpus:
        raise ValueError("Cannot tune LDA on an empty corpus.")
    measures = cfg.get("COHERENCE_MEASURES", ["c_v"])
    results = []
    for params in expand_param_grid(grid):
        trial_cfg = cfg["LDA"].copy()
        trial_cfg.update({k: v for k, v in params.items() if v is not None})
        t0 = time.time()
        model = train_lda(stage.corpus, stage.dictionary, trial_cfg)
        elapsed = time.time() - t0
        metrics, _ = compute_coherence_scores(
            model,
            stage.tokens,
            stage.dictionary,
            stage.corpus,
            measures,
        )
        perplexity = lda_perplexity(model, stage.corpus)
        row = {
            "params": params,
            "train_seconds": elapsed,
            "perplexity": perplexity,
        }
        for m in measures:
            row[f"coherence_{m}"] = metrics.get(m, float("nan"))
        results.append(row)
    return pd.DataFrame(results)


def grid_search_hdp(stage: PreprocessStageResult, cfg: Dict, grid: Dict[str, Iterable]):
    if stage.dictionary is None or len(stage.dictionary) == 0 or not stage.corpus:
        raise ValueError("Cannot tune HDP on an empty corpus.")
    measures = cfg.get("COHERENCE_MEASURES", ["c_v"])
    results = []
    base_cfg = cfg["HDP_2"].copy()
    for params in expand_param_grid(grid):
        trial_cfg = base_cfg.copy()
        trial_cfg.update(params)
        t0 = time.time()
        model = train_hdp(stage.corpus, stage.dictionary, trial_cfg)
        elapsed = time.time() - t0
        lda_like = as_lda(model)
        metrics, _ = compute_coherence_scores(
            lda_like,
            stage.tokens,
            stage.dictionary,
            stage.corpus,
            measures,
        )
        perplexity = lda_perplexity(lda_like, stage.corpus)
        row = {
            "params": params,
            "train_seconds": elapsed,
            "perplexity": perplexity,
        }
        for m in measures:
            row[f"coherence_{m}"] = metrics.get(m, float("nan"))
        results.append(row)
    return pd.DataFrame(results)



def _load_openai_client(api_key: Optional[str] = None, cfg: Optional[Dict[str, Any]] = None):
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise ImportError("Install openai>=1.0.0 to enable API interactions.") from exc

    cfg = cfg or CONFIG
    openai_cfg = cfg.get("OPENAI", {}) if isinstance(cfg, dict) else {}
    key = api_key or openai_cfg.get("API_KEY") or os.getenv("OPENAI_API_KEY")
    if not key:
        raise ValueError("OpenAI API key not provided via config or environment.")
    client_kwargs = {"api_key": key}
    org_id = openai_cfg.get("ORG_ID") or os.getenv("OPENAI_ORG_ID")
    project = openai_cfg.get("PROJECT") or os.getenv("OPENAI_PROJECT")
    if org_id:
        client_kwargs["organization"] = org_id
    if project:
        client_kwargs["project"] = project
    return OpenAI(**client_kwargs)






def prepare_topic_word_payload(stage_runs: Dict, limit: int = 50):
    payload = {}
    for (stage_name, model_key), run in stage_runs.items():
        if run.model is None or not run.top_words:
            continue
        trimmed = [words[:limit] for words in run.top_words]
        payload[f"{stage_name}:{run.label}"] = trimmed
    return payload


def classify_topics_with_openai(stage_runs: Dict, cfg: Dict, extra_instructions: Optional[str] = None, api_key: Optional[str] = None):
    payload = prepare_topic_word_payload(stage_runs, limit=cfg.get("OPENAI_MAX_WORDS_PER_MODEL", 50))
    if not payload:
        raise ValueError("No topic words available to send to OpenAI.")
    client = _load_openai_client(api_key=api_key, cfg=cfg)
    industries = cfg.get("INDUSTRY_LABELS", [])
    prompt_header = (
        "You are helping map topic models to industries. Use ONLY the provided industry list."
        "For each model/topic, assign the single best-fitting industry and provide a short rationale."
    )
    if extra_instructions:
        prompt_header += "" + extra_instructions
    prompt_header += "Industry list: " + ", ".join(industries) + ""

    sections = []
    for model_name, topics in payload.items():
        topic_lines = []
        for idx, words in enumerate(topics):
            topic_lines.append(f"Topic {idx}: {', '.join(words)}")
        sections.append(f"Model {model_name} " + "".join(topic_lines))
    prompt = prompt_header + "".join(sections)

    response = client.responses.create(
        model=cfg.get("OPENAI_MODEL", "gpt-4.1-mini"),
        input=prompt,
        temperature=cfg.get("OPENAI_TEMPERATURE", 0.2),
    )
    return getattr(response, "output_text", None) or response.to_dict()


def analyze_run_report_with_openai(report_path: Path, cfg: Dict, api_key: Optional[str] = None, extra_prompt: Optional[str] = None):
    client = _load_openai_client(api_key=api_key, cfg=cfg)
    with report_path.open("r", encoding="utf-8") as f:
        report_text = f.read()
    base_prompt = (
        "Summarize key findings, anomalies, and recommended follow-up actions from the topic-model run report below."
        "Focus on interpreting metrics, comparing preprocessing stages, and highlighting risks."
    )
    if extra_prompt:
        base_prompt += "" + extra_prompt
    prompt = base_prompt + "Report:" + report_text
    response = client.responses.create(
        model=cfg.get("OPENAI_MODEL", "gpt-4.1-mini"),
        input=prompt,
        temperature=cfg.get("OPENAI_TEMPERATURE", 0.2),
    )
    return getattr(response, "output_text", None) or response.to_dict()


def analyze_topics_excel_with_openai(df_topics: pd.DataFrame, cfg: Dict, manual_col: str = "Manual_Tag", api_key: Optional[str] = None, extra_prompt: Optional[str] = None):
    client = _load_openai_client(api_key=api_key, cfg=cfg)
    sample = df_topics.head(200).to_dict(orient="records")
    prompt = (
        "You will receive a JSON array with topic probabilities from multiple models and manual category labels."
        "Identify correlations between topic distributions across models and the manual tag."
    )
    if extra_prompt:
        prompt += "" + extra_prompt 
    prompt += "Respond with bullet points covering correlations, disagreements, and hypotheses."
    payload = json.dumps({"records": sample, "columns": df_topics.columns.tolist()})
    response = client.responses.create(
        model=cfg.get("OPENAI_MODEL", "gpt-4.1-mini"),
        input=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": payload},
        ],
        temperature=cfg.get("OPENAI_TEMPERATURE", 0.2),
    )
    return getattr(response, "output_text", None) or response.to_dict()


def plot_topic_heatmap(df: pd.DataFrame, topic_cols: List[str], manual_col: str, title: str, out_path: Path):
    if sns is None:
        print("seaborn is not installed; skipping heatmap generation.")
        return None
    if manual_col not in df.columns or not topic_cols:
        return None
    pivot = df.groupby(manual_col)[topic_cols].mean()
    if pivot.empty:
        return None
    plt.figure(figsize=(max(8, len(topic_cols) * 0.5), max(4, len(pivot) * 0.6)))
    sns.heatmap(pivot, cmap="viridis")
    plt.title(title)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.close()
    return out_path


def plot_top_topic_counts(df: pd.DataFrame, topic_cols: List[str], manual_col: str, title: str, out_path: Path):
    if manual_col not in df.columns or not topic_cols:
        return None
    topic_array = df[topic_cols].values
    if topic_array.size == 0:
        return None
    top_topic_idx = topic_array.argmax(axis=1)
    top_topic_labels = [topic_cols[i] for i in top_topic_idx]
    temp_df = pd.DataFrame({"Manual": df[manual_col].astype(str), "TopTopic": top_topic_labels})
    counts = temp_df.groupby(["Manual", "TopTopic"]).size().reset_index(name="Count")
    counts = counts.sort_values("Count", ascending=False).head(30)
    if counts.empty:
        return None
    counts_pivot = counts.pivot(index="Manual", columns="TopTopic", values="Count").fillna(0)
    plt.figure(figsize=(max(10, len(counts_pivot) * 0.6), 6))
    counts_pivot.plot(kind="bar", stacked=True, ax=plt.gca())
    plt.title(title)
    plt.ylabel("Document count")
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.close()
    return out_path


In [6]:
# --------------------- PIPELINE -----------------------

# 1) Load data
df_raw = load_json_dataset(CONFIG["JSON_PATH"])
df_raw["Company_norm"] = df_raw["Company"].map(normalize_name)

logger = PipelineLogger(CONFIG)
logger.log(f"Loaded dataset rows={len(df_raw)} columns={len(df_raw.columns)}")

# 2) Build preprocessing stages
cols_keep = ["Company", "Description", "Company_norm"]
stages = build_preprocessing_stages(df_raw[cols_keep], CONFIG, logger=logger)
stage_map = {stage.name: stage for stage in stages}

# 3) Train topic models for each stage
stage_runs, metrics_df, per_topic_df, top_words_df, log_lines = run_topic_models_across_stages(stages, CONFIG, logger=logger)
log_lines = logger.lines
print("Stage metrics shape:", metrics_df.shape)
print("Per-topic coherence shape:", per_topic_df.shape)
print("Top words rows:", len(top_words_df))

save_stage_outputs(metrics_df, per_topic_df, top_words_df, CONFIG)
logger.log(f"Stage metrics saved to {CONFIG['STAGE_METRICS_PATH']}")
logger.log(f"Per-topic coherence saved to {CONFIG['TOPIC_COHERENCE_PATH']}")
logger.log(f"Top words saved to {CONFIG['TOP_WORDS_PATH']}")

# 4) Select best model based on configured metric
best_metric = CONFIG.get("BEST_MODEL_METRIC", "c_v")
best_model_info = select_best_topic_model(metrics_df, metric=best_metric)
if best_model_info:
    score = best_model_info.get(f"coherence_{best_metric}", float("nan"))
    logger.log(
        f"Best model by {best_metric}: stage={best_model_info['stage']} model={best_model_info['model']} score={score}"
    )
else:
    logger.log("Best model could not be determined (check metrics file).", level="WARNING")

# 5) Pick final stage for downstream outputs
final_stage_name = CONFIG.get("FINAL_STAGE_NAME", stages[-1].name)
if final_stage_name not in stage_map:
    logger.log(
        f"Configured FINAL_STAGE_NAME '{final_stage_name}' not found; defaulting to last stage {stages[-1].name}."
    )
    final_stage_name = stages[-1].name
final_stage = stage_map[final_stage_name]
logger.log(f"Using stage '{final_stage_name}' for downstream exports.")

# 6) Build document-topic matrices for final stage
model_keys = {"LDA": "LDA", "HDP_1": "HDP1", "HDP_2": "HDP2"}
final_runs = {}
for mkey in model_keys.keys():
    run = stage_runs.get((final_stage_name, mkey))
    final_runs[mkey] = run
    if run is None or run.model is None:
        logger.log(f"{mkey} unavailable for final stage; document-topic matrix will be empty for this model.", level="WARNING")

doc_topic_mats = {}
inference_msgs = []
for mkey, run in final_runs.items():
    if run is None or run.model is None:
        continue
    t0 = time.time()
    matrix, topic_count = dense_doc_topics(run.lda_like, final_stage.corpus)
    infer_seconds = time.time() - t0
    doc_topic_mats[run.label] = {
        "matrix": matrix,
        "topics": topic_count,
        "run": run,
        "infer_seconds": infer_seconds,
    }
    inference_msgs.append(f"{run.label}: topics={topic_count} infer_s={infer_seconds:.2f}")
if inference_msgs:
    logger.log("Inference times (final stage): " + " | ".join(inference_msgs))
else:
    logger.log("No topic model available to compute document-topic matrices.", level="WARNING")

# 7) Assemble topics dataframe for final stage
topic_frames = []
topic_columns_map = {}
for label, info in doc_topic_mats.items():
    cols = [f"{label}_topic{i}" for i in range(info["topics"])]
    topic_frames.append(pd.DataFrame(info["matrix"], columns=cols))
    topic_columns_map[label] = cols

df_topics = pd.concat(topic_frames, axis=1) if topic_frames else pd.DataFrame()
df_proc = final_stage.df.reset_index(drop=True)[["Company", "Company_norm", "Description"]]
df_out = pd.concat([df_proc, df_topics.reset_index(drop=True)], axis=1)

# 8) Persist best-model doc-topic matrix (optional)
label_to_key = {"LDA": "LDA", "HDP1": "HDP_1", "HDP2": "HDP_2"}
if best_model_info:
    best_stage_name = best_model_info["stage"]
    best_label = best_model_info["model"]
    best_key = label_to_key.get(best_label)
    best_stage = stage_map.get(best_stage_name)
    best_run = stage_runs.get((best_stage_name, best_key)) if best_key else None
    if best_stage and best_run and best_run.model is not None:
        best_matrix, best_topics = dense_doc_topics(best_run.lda_like, best_stage.corpus)
        best_cols = [f"Best_{best_label}_topic{i}" for i in range(best_topics)]
        best_df = pd.concat(
            [
                best_stage.df.reset_index(drop=True)[["Company", "Company_norm"]],
                pd.DataFrame(best_matrix, columns=best_cols),
            ],
            axis=1,
        )
        best_path = CONFIG["OUTPUT_DIR"] / f"{CONFIG['RUN_PREFIX']}best_model_{best_stage_name}_{best_label}.csv"
        best_df.to_csv(best_path, index=False)
        logger.log(f"Best model document-topic matrix saved to {best_path}")
    else:
        logger.log("Best model matrix skipped (model or stage missing).", level="WARNING")

# 9) Optional hyperparameter tuning
if CONFIG.get("RUN_TOPIC_TUNING"):
    tuning_dir = CONFIG["OUTPUT_DIR"] / f"{CONFIG['RUN_PREFIX']}tuning"
    tuning_dir.mkdir(parents=True, exist_ok=True)
    try:
        lda_tuning_df = grid_search_lda(final_stage, CONFIG, CONFIG.get("LDA_TUNING_GRID", {}))
        if not lda_tuning_df.empty:
            lda_tuning_path = tuning_dir / f"{CONFIG['RUN_PREFIX']}lda_tuning.csv"
            lda_tuning_df.to_csv(lda_tuning_path, index=False)
            logger.log(f"LDA tuning results saved to {lda_tuning_path}")
    except Exception as exc:
        logger.log(f"LDA tuning skipped: {exc}")
    try:
        hdp_tuning_df = grid_search_hdp(final_stage, CONFIG, CONFIG.get("HDP_TUNING_GRID", {}))
        if not hdp_tuning_df.empty:
            hdp_tuning_path = tuning_dir / f"{CONFIG['RUN_PREFIX']}hdp_tuning.csv"
            hdp_tuning_df.to_csv(hdp_tuning_path, index=False)
            logger.log(f"HDP tuning results saved to {hdp_tuning_path}")
    except Exception as exc:
        logger.log(f"HDP tuning skipped: {exc}")

# 10) Merge manual tags
try:
    tags = load_manual_tags(
        CONFIG["MANUAL_TAG_PATH"],
        sheet=CONFIG["MANUAL_TAG_SHEET"],
        company_cands=CONFIG["MERGE_COMPANY_COLS"],
        label_cands=CONFIG["MERGE_LABEL_COLS"],
    )
    tags["Company_norm"] = tags["Company_Tag"].map(normalize_name)
    df_merged = df_out.merge(tags[["Company_norm", "Manual_Tag"]], on="Company_norm", how="left").drop(columns=["Company_norm"])
except Exception as exc:
    logger.log(f"Manual tag merge skipped: {exc}")
    df_merged = df_out.copy()

# 11) Supervised learning step
lda_cols = topic_columns_map.get("LDA", [])
hdp1_cols = topic_columns_map.get("HDP1", [])
hdp2_cols = topic_columns_map.get("HDP2", [])
topic_feature_cols = lda_cols + hdp1_cols + hdp2_cols
report_df = pd.DataFrame()
try:
    df_sup = df_merged.dropna(subset=["Manual_Tag"]).reset_index(drop=True)
    if topic_feature_cols and not df_sup.empty:
        X = df_sup[topic_feature_cols].values
        y = df_sup["Manual_Tag"].astype(str).values
        best_name, best_model, report_df, report_txt, cm, split, mask_used = pick_best_supervised(
            X, y, random_state=CONFIG["RANDOM_STATE"]
        )
        print("Best supervised model:", best_name)
        print("Validation metrics:", report_df)
        print("Classification report:", report_txt)
        print("Confusion matrix:", cm)
        logger.log(f"Best supervised classifier: {best_name}")
    else:
        print("Supervised step skipped: no topic features or labeled samples available.")
        logger.log("Supervised step skipped: insufficient data.", level="WARNING")
except ValueError as exc:
    print("Supervised step skipped:", exc)
    logger.log(f"Supervised step skipped: {exc}", level="WARNING")

# 12) Class-topic summaries
if "Manual_Tag" in df_merged.columns and not df_merged["Manual_Tag"].isna().all():
    if lda_cols:
        lda_summary = class_topic_summary(df_merged.dropna(subset=["Manual_Tag"]), lda_cols, "Manual_Tag", topn=5)
        lda_summary.to_csv(CONFIG["SAVE_CLASS_TOPIC_SUMMARY_CSV_PREFIX"] + "LDA.csv", index=False)
    if hdp1_cols:
        hdp1_summary = class_topic_summary(df_merged.dropna(subset=["Manual_Tag"]), hdp1_cols, "Manual_Tag", topn=5)
        hdp1_summary.to_csv(CONFIG["SAVE_CLASS_TOPIC_SUMMARY_CSV_PREFIX"] + "HDP.csv", index=False)
    if hdp2_cols:
        hdp2_summary = class_topic_summary(df_merged.dropna(subset=["Manual_Tag"]), hdp2_cols, "Manual_Tag", topn=5)
        hdp2_summary.to_csv(CONFIG["SAVE_CLASS_TOPIC_SUMMARY_CSV_PREFIX"] + "HDP2.csv", index=False)
    logger.log("Saved class-topic summaries for available models.")

# 13) Save merged outputs
if CONFIG.get("SAVE_TOPICS_CSV"):
    df_merged.to_csv(CONFIG["SAVE_TOPICS_CSV"], index=False)
    logger.log(f"Full topics CSV saved to {CONFIG['SAVE_TOPICS_CSV']}")
if not report_df.empty and CONFIG.get("SAVE_SUPERVISED_REPORT_CSV"):
    report_df.to_csv(CONFIG["SAVE_SUPERVISED_REPORT_CSV"], index=False)
    logger.log(f"Supervised report saved to {CONFIG['SAVE_SUPERVISED_REPORT_CSV']}")


if not df_sup.empty:
    subset_reports = []
    for subset_cfg in CONFIG.get("SUPERVISED_SUBSETS", []):
        labels = subset_cfg.get("labels") or []
        if not labels:
            continue
        subset_name = subset_cfg.get("name", "subset")
        min_docs = subset_cfg.get("min_docs", 0)
        df_subset = df_sup[df_sup["Manual_Tag"].isin(labels)].reset_index(drop=True)
        if df_subset.empty or len(df_subset) < max(min_docs, 2):
            logger.log(f"Subset '{subset_name}' skipped: not enough samples after filtering.", level="WARNING")
            continue
        try:
            X_sub = df_subset[topic_feature_cols].values
            y_sub = df_subset["Manual_Tag"].astype(str).values
            best_name, best_model, report_sub, report_txt_sub, cm_sub, _, _ = pick_best_supervised(
                X_sub, y_sub, random_state=CONFIG["RANDOM_STATE"]
            )
            report_sub.insert(0, "Dataset", subset_name)
            subset_reports.append(report_sub)
            logger.log(f"Subset '{subset_name}' best classifier: {best_name}")
            logger.log(f"Subset '{subset_name}' classification report:{report_txt_sub}")
        except ValueError as exc:
            logger.log(f"Subset '{subset_name}' skipped: {exc}", level="WARNING")
    if subset_reports:
        combined_subset = pd.concat(subset_reports, ignore_index=True)
        report_df = pd.concat([report_df, combined_subset], ignore_index=True) if not report_df.empty else combined_subset

# 14) OpenAI integrations (optional)
if CONFIG.get("ENABLE_OPENAI_CALLS"):
    openai_dir = CONFIG["OUTPUT_DIR"] / f"{CONFIG['RUN_PREFIX']}openai"
    openai_dir.mkdir(parents=True, exist_ok=True)
    try:
        classification_text = classify_topics_with_openai(stage_runs, CONFIG)
        class_path = openai_dir / f"{CONFIG['RUN_PREFIX']}topic_industry_mapping.txt"
        with class_path.open("w", encoding="utf-8") as f:
            f.write(classification_text if isinstance(classification_text, str) else json.dumps(classification_text, indent=2))
        logger.log(f"OpenAI topic classification saved to {class_path}")
    except Exception as exc:
        logger.log(f"Topic classification via OpenAI failed: {exc}", level="WARNING")
    try:
        report_analysis = analyze_run_report_with_openai(Path(CONFIG["RUN_LOG_PATH"]), CONFIG)
        report_path = openai_dir / f"{CONFIG['RUN_PREFIX']}report_analysis.txt"
        with report_path.open("w", encoding="utf-8") as f:
            f.write(report_analysis if isinstance(report_analysis, str) else json.dumps(report_analysis, indent=2))
        logger.log(f"OpenAI run-report analysis saved to {report_path}")
    except Exception as exc:
        logger.log(f"Run-report analysis via OpenAI failed: {exc}", level="WARNING")
    try:
        topics_analysis = analyze_topics_excel_with_openai(df_merged, CONFIG)
        topics_path = openai_dir / f"{CONFIG['RUN_PREFIX']}topics_manual_tag_analysis.txt"
        with topics_path.open("w", encoding="utf-8") as f:
            f.write(topics_analysis if isinstance(topics_analysis, str) else json.dumps(topics_analysis, indent=2))
        logger.log(f"OpenAI topic-probability analysis saved to {topics_path}")
    except Exception as exc:
        logger.log(f"Topic probability analysis via OpenAI failed: {exc}", level="WARNING")

# 15) Visualisations
if CONFIG.get("ENABLE_VISUALS") and "Manual_Tag" in df_merged.columns and not df_merged["Manual_Tag"].isna().all():
    visuals_dir = Path(CONFIG["VISUALS_DIR"])
    for label, cols in topic_columns_map.items():
        if not cols:
            continue
        heatmap_path = visuals_dir / f"{CONFIG['RUN_PREFIX']}{label}_manual_heatmap.png"
        plot_topic_heatmap(df_merged, cols, "Manual_Tag", f"{label} mean topic weight by manual tag", heatmap_path)
        counts_path = visuals_dir / f"{CONFIG['RUN_PREFIX']}{label}_manual_top_topic.png"
        plot_top_topic_counts(df_merged, cols, "Manual_Tag", f"{label} dominant topic vs manual tag", counts_path)
    logger.log(f"Visualisations saved under {visuals_dir}")

# 16) Persist log file
write_log(logger.lines, Path(CONFIG["RUN_LOG_PATH"]))
print("Log written to", CONFIG["RUN_LOG_PATH"])

print("Output rows:", len(df_merged), " | Columns:", len(df_merged.columns))
df_merged.head(3)

[2025-09-21 20:55:40] [INFO] Loaded dataset rows=3195 columns=3
[2025-09-21 21:06:04] [INFO] [Stage tokenized] Tokenized (lowercased, accent stripped) | documents=3195, non_empty_docs=3071, total_tokens=309305, avg_tokens=100.7180, median_tokens=81.0000, min_tokens=0, max_tokens=250, std_tokens=64.0743, unique_tokens=21143, vocab_size=4230


c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\gensim\models\ldamodel.py:850: RuntimeWarning: overflow encountered in exp2
  perwordbound, np.exp2(-perwordbound), len(chunk), corpus_words


[2025-09-21 21:10:06] [INFO] [Stage tokenized::LDA] train_s=7.11s perplexity=1160117339211003526150279804592620668655260419413475789567761978287328381315437979356563700807238528849898633800070392612216584942911488.0000 coherence_c_v=0.3752 coherence_u_mass=-1.8229 coherence_c_npmi=0.0026 coherence_c_uci=-0.3301 | train_seconds=7.1140, perplexity=1160117339211003526150279804592620668655260419413475789567761978287328381315437979356563700807238528849898633800070392612216584942911488.0000, doc_count=3195, dictionary_size=4230, coherence_c_v=0.3752, coherence_u_mass=-1.8229, coherence_c_npmi=0.0026, coherence_c_uci=-0.3301
[2025-09-21 21:10:06] [INFO] [Stage tokenized::HDP1] train_s=0.91s perplexity=783.3916 coherence_c_v=0.7086 coherence_u_mass=-18.5970 coherence_c_npmi=-0.4291 coherence_c_uci=-11.9762 | train_seconds=0.9064, perplexity=783.3916, doc_count=3195, dictionary_size=4230, coherence_c_v=0.7086, coherence_u_mass=-18.5970, coherence_c_npmi=-0.4291, coherence_c_uci=-11.9762
[2025-0

c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sanie.s.rojas.lobo\Desktop\

Best supervised model: GradientBoosting
Validation metrics:                 Model   MacroF1  Accuracy
0    GradientBoosting  0.328144  0.500000
1     LogReg_balanced  0.311966  0.461538
2        RandomForest  0.295621  0.538462
3  LinearSVC_balanced  0.271062  0.384615
Classification report:                                         precision    recall  f1-score   support

                   Aerospace & Defense     0.4000    0.5000    0.4444         4
                 Automotive & Assembly     0.0000    0.0000    0.0000         1
                             Chemicals     1.0000    1.0000    1.0000         1
               Consumer Packaged Goods     0.0000    0.0000    0.0000         1
          Electric Power & Natural Gas     0.5000    1.0000    0.6667         1
                    Financial Services     0.5000    0.5000    0.5000         2
                            Healthcare     0.2000    0.3333    0.2500         3
             Industrials & Electronics     0.6667    0.5000    0.5

c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[2025-09-21 21:32:48] [INFO] Subset 'priority_industries' best classifier: GradientBoosting
[2025-09-21 21:32:48] [INFO] Subset 'priority_industries' classification report:                     precision    recall  f1-score   support

Aerospace & Defense     1.0000    0.7500    0.8571         4
 Financial Services     0.5000    0.5000    0.5000         2
         Healthcare     1.0000    1.0000    1.0000         3
             Retail     0.8333    1.0000    0.9091         5

           accuracy                         0.8571        14
          macro avg     0.8333    0.8125    0.8166        14
       weighted avg     0.8690    0.8571    0.8553        14
Log written to artifacts\20250921-205356_pipeline_report.txt
Output rows: 2477  | Columns: 153


,Company,Description,LDA_topic0,LDA_topic1,LDA_topic2,LDA_topic3,LDA_topic4,LDA_topic5,LDA_topic6,LDA_topic7,...,HDP2_topic91,HDP2_topic92,HDP2_topic93,HDP2_topic94,HDP2_topic95,HDP2_topic96,HDP2_topic97,HDP2_topic98,HDP2_topic99,Manual_Tag
0,3D Systems,3D Systems Corporation is an American company ...,0.020934,0.011054,0.143382,0.166610,0.026417,0.009154,0.021657,0.032201,...,0.000003,0.000003,0.000002,0.000003,0.000004,0.000002,0.000001,0.000002,8.887604e-07,Industrials & Electronics
1,3M,The 3M Company (originally the Minnesota Minin...,0.022157,0.100907,0.139593,0.023606,0.012217,0.042196,0.025281,0.022567,...,0.000003,0.000003,0.000002,0.000003,0.000004,0.000002,0.000001,0.000002,8.430951e-07,Consumer Packaged Goods
2,A. O. Smith,A. O. Smith Corporation is an American manufac...,0.015227,0.083529,0.119695,0.035482,0.014049,0.178143,0.015815,0.029123,...,0.000005,0.000005,0.000004,0.000006,0.000008,0.000003,0.000002,0.000004,1.606731e-06,Industrials & Electronics


In [8]:
# 11a) Gradient boosting subset comparison
gradient_boosting_subset_results = []
gradient_boosting_subset_reports = {}
gradient_boosting_subset_confusions = {}

df_sup_available = 'df_sup' in globals() and 'df_sup' in locals()

if not df_sup_available or df_sup.empty:
    print("Gradient boosting evaluation skipped: no labeled data in df_sup.")
else:
    metric_name = CONFIG.get("BEST_MODEL_METRIC", "c_v")
    metric_col = f"coherence_{metric_name}"
    if metrics_df.empty or metric_col not in metrics_df.columns:
        print(f"Gradient boosting evaluation skipped: metrics for '{metric_col}' unavailable.")
    else:
        model_pairs = [
            ("LDA", "LDA"),
            ("HDP1", "HDP_1"),
            ("HDP2", "HDP_2"),
        ]
        manual_lookup = (
            df_sup[["Company", "Manual_Tag"]]
            .dropna()
            .assign(Company_norm=lambda d: d["Company"].map(normalize_name))
            .set_index("Company_norm")["Manual_Tag"]
            .to_dict()
        )
        for model_label, model_key in model_pairs:
            df_metrics_model = (
                metrics_df[metrics_df["model"] == model_label]
                .dropna(subset=[metric_col])
            )
            if df_metrics_model.empty:
                logger.log(f"Gradient boosting skipped for {model_label}: no metrics available.", level='WARNING')
                continue
            best_idx = df_metrics_model[metric_col].astype(float).idxmax()
            best_stage = df_metrics_model.loc[best_idx, "stage"]
            stage_result = stage_map.get(best_stage)
            run = stage_runs.get((best_stage, model_key))
            if stage_result is None or run is None or run.model is None:
                logger.log(
                    f"Gradient boosting skipped for {model_label}: stage {best_stage} run not available.",
                    level='WARNING',
                )
                continue
            matrix, topic_count = dense_doc_topics(run.lda_like, stage_result.corpus)
            if topic_count == 0 or matrix.size == 0:
                logger.log(
                    f"Gradient boosting skipped for {model_label}: empty topic matrix at stage {best_stage}.",
                    level='WARNING',
                )
                continue
            stage_df = stage_result.df.reset_index(drop=True).copy()
            if "Company_norm" not in stage_df.columns:
                stage_df["Company_norm"] = stage_df["Company"].map(normalize_name)
            manual_series = stage_df["Company_norm"].map(manual_lookup)
            mask_labeled = manual_series.notna()
            if mask_labeled.sum() < 5:
                logger.log(
                    f"Gradient boosting skipped for {model_label}: insufficient labeled samples at stage {best_stage}.",
                    level='WARNING',
                )
                continue
            topic_cols = [f"{model_label}_topic{i}" for i in range(topic_count)]
            topics_labeled = matrix[mask_labeled.values]
            y_labeled = manual_series[mask_labeled].astype(str).values
            df_topics_labeled = pd.DataFrame(topics_labeled, columns=topic_cols)
            df_topics_labeled["Manual_Tag"] = y_labeled

            for subset_cfg in CONFIG.get("SUPERVISED_SUBSETS", []):
                subset_labels = subset_cfg.get("labels") or []
                if not subset_labels:
                    continue
                subset_name = subset_cfg.get("name", "subset")
                min_docs = subset_cfg.get("min_docs", 0)
                df_subset = df_topics_labeled[df_topics_labeled["Manual_Tag"].isin(subset_labels)].reset_index(drop=True)
                if df_subset.empty or len(df_subset) < max(min_docs, 2):
                    logger.log(
                        f"Gradient boosting skipped for {model_label} on subset '{subset_name}': insufficient samples after filtering.",
                        level='WARNING',
                    )
                    continue

                X_subset = df_subset[topic_cols].values
                y_subset = df_subset["Manual_Tag"].astype(str).values
                X_adj, y_adj, info, _ = filter_labels_min_count(
                    X_subset,
                    y_subset,
                    min_count=2,
                    merge_to_other=False,
                )
                if info:
                    logger.log(
                        f"Gradient boosting label adjustment for {model_label} on subset '{subset_name}': {info}"
                    )
                unique_labels = np.unique(y_adj)
                if len(unique_labels) < 2 or len(y_adj) < max(10, len(unique_labels) * 2):
                    logger.log(
                        f"Gradient boosting skipped for {model_label} on subset '{subset_name}': not enough classes post-filtering.",
                        level='WARNING',
                    )
                    continue
                try:
                    X_train, X_test, y_train, y_test = train_test_split(
                        X_adj,
                        y_adj,
                        test_size=CONFIG.get("TEST_SIZE", 0.2),
                        random_state=CONFIG.get("RANDOM_STATE", 42),
                        stratify=y_adj,
                    )
                except ValueError as exc:
                    logger.log(
                        f"Gradient boosting split failed for {model_label} on subset '{subset_name}': {exc}",
                        level='WARNING',
                    )
                    continue

                gb_clf = GradientBoostingClassifier(random_state=CONFIG.get("RANDOM_STATE", 42))
                gb_clf.fit(X_train, y_train)
                y_pred = gb_clf.predict(X_test)
                macro_f1 = f1_score(y_test, y_pred, average="macro")
                accuracy = accuracy_score(y_test, y_pred)

                gradient_boosting_subset_results.append(
                    {
                        "Subset": subset_name,
                        "Model": model_label,
                        "Stage": best_stage,
                        "Topics": topic_count,
                        "TrainSize": len(y_train),
                        "TestSize": len(y_test),
                        "MacroF1": macro_f1,
                        "Accuracy": accuracy,
                    }
                )
                gradient_boosting_subset_reports[(subset_name, model_label)] = classification_report(
                    y_test,
                    y_pred,
                    digits=4,
                )
                gradient_boosting_subset_confusions[(subset_name, model_label)] = confusion_matrix(y_test, y_pred)

        gradient_boosting_subset_results_df = pd.DataFrame(gradient_boosting_subset_results)
        if not gradient_boosting_subset_results_df.empty:
            gradient_boosting_subset_results_df = (
                gradient_boosting_subset_results_df
                .sort_values(["Subset", "Model"])
                .reset_index(drop=True)
            )
            print("Gradient Boosting performance per subset/model:")
            print(gradient_boosting_subset_results_df.to_string(index=False))
            for (subset_name, model_label), report in gradient_boosting_subset_reports.items():
                print(f"Gradient Boosting classification report - subset={subset_name} model={model_label}:")
                print(report)
        else:
            print("Gradient boosting evaluation produced no results.")


Gradient Boosting performance per subset/model:
             Subset Model        Stage  Topics  TrainSize  TestSize  MacroF1  Accuracy
priority_industries  HDP1 no_stopwords      25         54        14 0.250000  0.428571
priority_industries  HDP2 no_stopwords     100         54        14 0.190909  0.214286
priority_industries   LDA pos_filtered      25         54        14 0.600000  0.642857
Gradient Boosting classification report - subset=priority_industries model=LDA:
                     precision    recall  f1-score   support

Aerospace & Defense     0.5000    0.5000    0.5000         4
 Financial Services     1.0000    0.5000    0.6667         2
         Healthcare     0.5000    0.3333    0.4000         3
             Retail     0.7143    1.0000    0.8333         5

           accuracy                         0.6429        14
          macro avg     0.6786    0.5833    0.6000        14
       weighted avg     0.6480    0.6429    0.6214        14

Gradient Boosting classification 

c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sanie.s.rojas.lobo\Desktop\ITBA Bucket\Tesis 2\TesisCode2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio